# Day 22 — Paper Selection & Research Entry

## Week 4: Research Training

**Main paper:** **Dynamic Bundling with Large Language Models for Zero-Shot Inference on Text-Attributed Graphs (DENSE)**  
Yusheng Zhao, Qixin Zhang, Xiao Luo, Weizhi Zhang, Zhiping Xiao, Wei Ju, Philip S. Yu, Ming Zhang — NeurIPS 2025.

### Today's goal
Today is **not** a full paper-reading day and **not** a coding day. By the end of Day 22, you should be able to explain why this is a good Week 4 entry point, what problem it studies, where it sits in Graph × LLM, what you expect its pipeline to be, and what Week 4 should verify.

$$
\text{Paper} \rightarrow \text{Claim} \rightarrow \text{Repo} \rightarrow
\text{Baseline} \rightarrow \text{Controlled Experiment} \rightarrow \text{Evidence}
$$

## 1. Resources

### Main paper — DENSE
- NeurIPS: https://papers.nips.cc/paper_files/paper/2025/hash/e4343147340c9d65f4c780451eb066f9-Abstract-Conference.html
- arXiv: https://arxiv.org/abs/2505.17599
- Official code: https://github.com/YushengZhao/bundle-neurips25
- Wei Ju publications: https://juweipku.github.io/publications/

### Auxiliary paper — HEAL
- ACL Anthology: https://aclanthology.org/2025.findings-emnlp.360/
- PDF: https://aclanthology.org/2025.findings-emnlp.360.pdf

### Reference — LLM Agent Survey
- arXiv: https://arxiv.org/abs/2503.21460
- Paper collection: https://github.com/luo-junyu/Awesome-Agent-Papers

### Future post-training branch — SemiEvol
- ACL Anthology: https://aclanthology.org/2025.findings-naacl.151/
- Official code: https://github.com/luo-junyu/SemiEvol

> Week 4 focuses on **DENSE**. HEAL is the auxiliary comparison paper. The survey is reference material; SemiEvol is saved for the post-training branch.

## 2. Why DENSE?

Your preparation already gives you most prerequisites:

**Week 2:** Graph → GCN / GraphSAGE / GAT  
**Week 3:** LLM → Agent → Graph × LLM → Graph RAG

DENSE sits at the intersection:

$$
\text{Text-Attributed Graph} + \text{LLM supervision} + \text{GNN}
\rightarrow \text{Zero-shot graph inference}
$$

This week is therefore a bridge from **learning graph/LLM concepts** to **understanding and testing a real Graph × LLM research system**.

## 3. First-pass vocabulary

**Text-Attributed Graph (TAG):** nodes have textual attributes in addition to graph relations.

```text
Paper node
├── graph relation: cites / cited by
└── text attribute: title + abstract
```

**Zero-shot inference:** solving the target task without ordinary task-specific labeled training examples in the usual supervised setting.

**Bundle:** instead of querying one isolated node text, DENSE groups related node texts.

**Bundle-level supervision:** the LLM response supervises a group/bundle rather than serving only as an isolated per-node prediction.

**Refinement:** potentially noisy items in bundles are filtered/refined during optimization.

## 4. Before reading: reconstruct the problem yourself

### Q1
Why might an LLM perform poorly if we give it only the text of one node from a graph?

**Your answer:** 

> Because the LLM can only observe the textual information of the target node, while the relationships and structural information between nodes are missing. Such graph structure may provide useful context that cannot be inferred from the node text alone.

### Q2
Suppose the graph is a citation network. What useful information could neighboring nodes provide that the target node text alone does not provide?

**Your answer:** 

> In a citation network, neighboring papers may contain related topics, methods, or research areas. Citation relationships therefore provide structural context that can help infer the semantic category of the target paper even when its own text is ambiguous or incomplete.

### Q3
Why might querying several structurally related node texts together be better than querying each node independently?

Do **not** answer only “because there is more information.” Be more precise.

**Your answer:**

> Structurally related nodes may contain complementary and correlated semantic information. Querying them together allows the LLM to use graph-guided context rather than making predictions from isolated node texts, which may produce more reliable supervision for the downstream graph model.

## 5. Predict the pipeline before reading Method

Fill in the missing steps.

```text
Text-Attributed Graph
        ↓
[ Step A: use graph structure to select structurally related nodes and construct bundles ]
        ↓
Bundles of related node texts
        ↓
[ Step B: query the LLM with each bundle to obtain bundle-level labels / supervision ]
        ↓
Bundle-level labels / supervision
        ↓
[ Step C: use the bundle-level supervision to train the GNN ]
        ↓
Graph model
        ↓
Zero-shot node prediction
```

**Which component uses graph structure?**

> step A

**Which component uses the LLM?**

> step B

**Which component learns from the resulting supervision?**

> step C / the GNN

## 6. Map DENSE onto what you already know

| Concept in DENSE | Closest concept from Weeks 2–3 | Same thing or different? |
|---|---|---|
| GNN | GCN / GraphSAGE / GAT | |
| Graph topology | neighborhood / message passing | |
| LLM query | LLM for Graph | |
| Bundle | subgraph / neighborhood evidence? | |
| Refinement | retrieval/noise control? | |
| Zero-shot inference | | |

Do not force equivalence. Identify where your previous concepts transfer and where DENSE introduces something new.

## 7. Research decomposition — Version 0

$$
\text{Task} \rightarrow \text{Bottleneck} \rightarrow \text{Hypothesis}
\rightarrow \text{Intervention} \rightarrow \text{Experiment} \rightarrow \text{Evidence}
$$

**Task**

> complete Zero-Shot Inference on Text-Attributed Graphs

**Bottleneck**

> limited information on graph structure and unreliable responses

**Hypothesis**

> Grouping structurally related node texts may provide richer graph-aware context to the LLM and produce more reliable supervision than querying nodes individually.

**Intervention**

> Introduce graph-guided text bundling, query the LLM for bundle-level labels, and use the generated supervision to train a GNN, together with refinement for noisy bundle items.

**What evidence would convince you that the method really works?**

> DENSE should outperform strong zero-shot baselines across multiple datasets. More importantly, controlled ablations should show that graph-guided bundling performs better than random bundling under the same context/token budget, and removing refinement should reduce performance if refinement is truly useful.

## 8. Do not trust the method yet

### Q4
If DENSE beats an individual-node LLM baseline, does that automatically prove graph structure is responsible?

> No. Better performance than an individual-node LLM baseline does not prove that graph structure is the cause, because the improvement may simply come from providing more textual context. We need controlled ablations that isolate the effect of graph-guided bundling.

### Q5
Design one control/ablation that distinguishes:

```text
"Graph structure helps"
```

from

```text
"The LLM simply received more text/context"
```

**Your proposed experiment:**

> Control all other variables and compare graph-guided bundles with randomly constructed bundles of the same size and token budget. If graph-guided bundles consistently achieve better performance, this provides stronger evidence that graph structure contributes beyond simply giving the LLM more text.

## 9. Repo reconnaissance — only 10 minutes

Open the official repository, but **do not install or run it today**.

Record:
- Main entry file: bundle.py
- Dataset location: detaset/
- Main GNN choices: gcn / sage / gin / glognn
- Important command-line arguments: device / dataset / bundle_size / num_samples / sample_criterion / max_hop / query_type / model / loss_type / gnn_type / stages / lr / wd /
- External API/model requirement: OpenAI API
- One thing that may make reproduction difficult: The reproduction may depend on external LLM API access, which introduces cost and possible output/version differences.

### Prediction
Which parameter could become a useful single-variable experiment later?

> bundle_size

I predict that bundle size could be a useful single-variable experiment because increasing it may provide richer structural context to the LLM, but an excessively large bundle may also introduce irrelevant or noisy information.

Possible candidates to inspect later: bundle size, graph hop range, GNN type, sampling criterion, refinement-related settings. Do not change anything today.

## 10. Auxiliary paper: HEAL — 15 minute comparison

Read only the **title + abstract** of HEAL.

### Q6
Why is HEAL closer to the phrase **Graph × LLM + Agent** than DENSE?

> Because HEAL explicitly uses LLM-based agents to enhance text-attributed hypergraphs, so it directly combines graph structure, LLMs, and agent-based interaction.

### Q7
Why might HEAL nevertheless be a worse *first reproduction paper* for you this week?

> Because HEAL introduces additional concepts such as hypergraphs, self-supervised learning, and multiple LLM-based agents, making the pipeline more complex than DENSE for a first reproduction task.

### Q8
Complete:

```text
DENSE: LLM helps graph learning by providing bundle-level supervision for zero-shot inference on text-attributed graphs.
HEAL: LLM-based agents help enhance incomplete text-attributed hypergraphs for self-supervised representation learning.
```

## 11. Day 22 final output

Write **5–8 sentences** answering:

> **Why did I choose DENSE as my first research-training paper, what problem does it solve, and what do I want to verify during Week 4?**

**Your answer:**

>

## 12. Questions to carry into Day 23

Do not solve these today.

1. What exactly are the two failure modes identified by the authors?
2. How is a bundle constructed?
3. Why should structural proximity make bundled supervision useful?
4. What does the LLM actually output?
5. How are bundle labels converted into GNN supervision?
6. Why is refinement necessary?
7. What are the strongest baselines?
8. Which experiments support each major claim?
9. Does the paper prove graph structure helps, or only that the full method helps?
10. Which component is the best candidate for our controlled experiment?

Tomorrow: **Paper Pass 1 — Problem → Gap → Method → Result**.

# Day 22 Completion Checklist

- [x] Opened the DENSE paper and official repository
- [x] Read title + abstract carefully
- [x] Understood TAG / zero-shot / bundle / bundle-level supervision
- [x] Predicted the pipeline before reading Method
- [x] Completed Q1–Q5
- [x] Did a 10-minute repo reconnaissance
- [x] Read only HEAL title + abstract for comparison
- [x] Completed Q6–Q8
- [x] Wrote the 5–8 sentence Day 22 research-entry summary
- [x] Did **not** start blindly installing/running the repository

**Stop here. Day 23 is the first real paper-reading pass.**